# Analyse de la fiabilité des prévisions météo

**YENER Dogukan — M2 DS2E, Université de Strasbourg, 2025–2026**  
GitHub : [github.com/studentyener](https://github.com/studentyener)

---

Ce notebook explore une question simple : dans quelle mesure les prévisions météo à quelques jours sont-elles fiables ?

Le pipeline compare des prévisions météo (collectées via API ou simulées) à des observations, et calcule des indicateurs d'erreur pour 8 villes françaises. L'objectif n'est pas de refaire la météo, mais de quantifier l'écart entre ce qui est prévu et ce qui se passe réellement.

**Plan du notebook :**
1. Mise en place et imports
2. Chargement du panel de villes
3. Collecte des prévisions
4. Construction de l'historique d'évaluation
5. Calcul des métriques de fiabilité
6. Visualisations
7. Analyse statistique
8. Export des résultats
9. Bilan

> **Reproductibilité :** sans clé API `OPENWEATHER_API_KEY`, tout tourne en mode synthétique. Les résultats sont identiques d'une exécution à l'autre (seed fixé).

---
## 1. Mise en place

In [ ]:
from pathlib import Path
import sys
import os

# Ajout du dossier src au path Python
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Modules du projet
from data_collection import build_cities_df, build_coords_df, collect_forecasts
from analysis import build_synthetic_history, compute_metrics, city_scores
from visualization import build_horizon_chart, build_heatmap
from config import DATA_PROCESSED_DIR, OUTPUTS_DIR

# Bibliothèques
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
print("Prêt.", datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Racine projet :", PROJECT_ROOT)

---
## 2. Panel de villes

J'ai retenu 8 villes françaises avec des profils géographiques variés : nord, littoral méditerranéen, Atlantique, intérieur continental, est. Les coordonnées sont fixées directement dans le code pour éviter toute dépendance réseau.

In [ ]:
villes_df = build_cities_df()
coords_df = build_coords_df(villes_df)

print(f"{len(villes_df)} villes dans le panel :")
display(villes_df)

In [ ]:
# Carte du panel
fig_map = px.scatter_mapbox(
    villes_df,
    lat="lat", lon="lon",
    hover_name="ville",
    hover_data={"lat": ":.4f", "lon": ":.4f", "pays": True},
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="carto-positron",
    title="Panel des villes analysées"
)
fig_map.update_traces(marker=dict(size=14, color="#2563EB"))
fig_map.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, height=450)
fig_map.show()

---
## 3. Collecte des prévisions

La fonction `collect_forecasts()` regarde si la variable `OPENWEATHER_API_KEY` est définie dans l'environnement. Si oui, elle appelle l'API OpenWeatherMap (endpoint `/forecast`, prévisions 3h sur 5 jours). Sinon, elle génère des données synthétiques avec le même schéma.

In [ ]:
api_key = os.getenv("OPENWEATHER_API_KEY", "").strip()
print("Mode :", "API réelle (OpenWeatherMap)" if api_key else "Données synthétiques (pas de clé API)")

df_forecasts = collect_forecasts(coords_df)

print(f"\nPrévisions collectées : {len(df_forecasts):,} lignes")
print(f"Villes                : {df_forecasts['ville'].nunique()}")
print(f"Horizons (h)          : {df_forecasts['horizon_h'].min()} → {df_forecasts['horizon_h'].max()}")
print(f"Source                : {df_forecasts['source'].unique()}")

df_forecasts.head(6)

In [ ]:
# Température prévue sur les 24 premières heures par ville
df_24 = df_forecasts[df_forecasts["horizon_h"] <= 24]

fig_fcst = px.line(
    df_24, x="horizon_h", y="temperature_c", color="ville",
    markers=True,
    title="Températures prévues — 24 prochaines heures",
    labels={"horizon_h": "Horizon (h)", "temperature_c": "Température (°C)", "ville": "Ville"}
)
fig_fcst.update_layout(template="plotly_white")
fig_fcst.show()

---
## 4. Historique d'évaluation

Pour mesurer la fiabilité, il faut comparer des prévisions à des observations réelles. Sans historique réel collecté sur plusieurs semaines, j'utilise un générateur synthétique qui reproduit les structures attendues :
- **effet latitude** : −0.35 °C par degré vers le nord
- **composante saisonnière** : sinusoïde sur l'année
- **biais de ville** : décalage aléatoire fixe propre à chaque ville
- **bruit gaussien** indépendant pour l'observation et la prévision

Le `seed=42` garantit que les résultats sont identiques à chaque exécution.

In [ ]:
df_historique = build_synthetic_history(villes_df, days=30, seed=42)

print(f"Historique : {len(df_historique):,} entrées")
print(f"Période    : {df_historique['date'].min().date()} → {df_historique['date'].max().date()}")

df_historique.head(8)

In [ ]:
# Prévision vs observation pour une ville
ville_ex = "Lyon"
df_ex = df_historique[df_historique["ville"] == ville_ex]

fig_ex = go.Figure()
fig_ex.add_trace(go.Scatter(
    x=df_ex["date"], y=df_ex["observed_temp_c"],
    name="Observation", line=dict(color="#16A34A", width=2)
))
fig_ex.add_trace(go.Scatter(
    x=df_ex["date"], y=df_ex["forecast_temp_c"],
    name="Prévision", line=dict(color="#DC2626", width=2, dash="dash")
))
fig_ex.update_layout(
    title=f"{ville_ex} — prévision vs observation (30 derniers jours)",
    xaxis_title="Date", yaxis_title="Température (°C)",
    template="plotly_white", legend_title=""
)
fig_ex.show()

---
## 5. Métriques de fiabilité

Trois indicateurs classiques pour évaluer une prévision numérique :

| Indicateur | Ce qu'il mesure |
|---|---|
| **MAE** (Mean Absolute Error) | Erreur moyenne en °C — facile à interpréter |
| **RMSE** (Root Mean Squared Error) | Même chose mais pénalise davantage les grosses erreurs |
| **Biais** | Surestimation (+) ou sous-estimation (−) systématique |

J'ajoute un **indice de fiabilité composite** : `100 / (1 + MAE + 0.5×RMSE + |biais|)` — plus c'est élevé, mieux c'est.

Et un **score final** sur 100 : `100 − (MAE×18 + RMSE×10 + |biais|×8)` — permet de classer les villes d'un coup d'œil.

In [ ]:
df_metrics = compute_metrics(df_historique)

print("Métriques par ville :")
df_metrics[
    ["ville", "mae", "rmse", "bias", "reliability_index", "observations"]
].sort_values("reliability_index", ascending=False).style \
    .format({"mae": "{:.2f}", "rmse": "{:.2f}", "bias": "{:+.2f}", "reliability_index": "{:.2f}"}) \
    .background_gradient(subset=["reliability_index"], cmap="RdYlGn")

In [ ]:
df_scores = city_scores(df_metrics)

print("Classement final :")
df_scores[["ville", "score", "mae", "rmse", "bias"]].style \
    .format({"score": "{:.1f}", "mae": "{:.2f}", "rmse": "{:.2f}", "bias": "{:+.2f}"}) \
    .background_gradient(subset=["score"], cmap="RdYlGn")

---
## 6. Visualisations

### 6.1 MAE par ville

In [ ]:
fig1 = build_horizon_chart(df_metrics)
fig1.update_layout(template="plotly_white", showlegend=False)
fig1.show()

### 6.2 Distribution des erreurs

In [ ]:
df_err = df_historique.copy()
df_err["erreur"] = df_err["forecast_temp_c"] - df_err["observed_temp_c"]

# Tri par MAE croissant
ordre = df_metrics.sort_values("mae")["ville"].tolist()

fig_box = px.box(
    df_err, x="ville", y="erreur", color="ville",
    category_orders={"ville": ordre},
    points="all",
    title="Distribution des erreurs de prévision par ville",
    labels={"erreur": "Erreur (°C)", "ville": ""}
)
fig_box.add_hline(y=0, line_dash="dash", line_color="black",
                  annotation_text="Erreur nulle")
fig_box.update_layout(template="plotly_white", showlegend=False)
fig_box.show()

# Bilan rapide
biais_max = df_metrics.loc[df_metrics["bias"].abs().idxmax()]
print(f"Biais le plus fort : {biais_max['ville']} ({biais_max['bias']:+.2f} °C)")
print("Un biais positif = la prévision surestime systématiquement la température.")

### 6.3 Heatmap multi-indicateurs

In [ ]:
fig2 = build_heatmap(df_metrics)
fig2.update_layout(height=380)
fig2.show()

print("Lecture : MAE et RMSE → faible = bien. reliability_index → élevé = bien.")

### 6.4 Classement final

In [ ]:
fig_rank = px.bar(
    df_scores, x="score", y="ville",
    orientation="h",
    color="score",
    color_continuous_scale="RdYlGn",
    range_color=[40, 70],
    title="Classement des villes — score de fiabilité (0–100)",
    labels={"score": "Score", "ville": ""},
    text="score"
)
fig_rank.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig_rank.update_layout(
    yaxis={"categoryorder": "total ascending"},
    template="plotly_white",
    coloraxis_showscale=False
)
fig_rank.show()

top = df_scores.iloc[0]
bot = df_scores.iloc[-1]
print(f"Meilleure : {top['ville']} — score {top['score']:.1f}, MAE {top['mae']:.2f} °C")
print(f"Moins bonne : {bot['ville']} — score {bot['score']:.1f}, MAE {bot['mae']:.2f} °C")

---
## 7. Analyse statistique

### 7.1 Normalité des erreurs (test de Shapiro-Wilk)

L'utilisation du MAE et du RMSE ne suppose pas nécessairement la normalité, mais c'est utile de vérifier la distribution des erreurs pour savoir si des tests paramétriques sont applicables par la suite.

In [ ]:
print("Test de Shapiro-Wilk — H0 : les erreurs suivent une loi normale (α = 0.05)\n")

resultats = []
for v in df_err["ville"].unique():
    e = df_err[df_err["ville"] == v]["erreur"].values
    w, p = stats.shapiro(e)
    resultats.append({
        "Ville": v,
        "W": round(w, 4),
        "p-valeur": round(p, 4),
        "Normalité": "✓" if p > 0.05 else "✗"
    })

df_sw = pd.DataFrame(resultats).sort_values("p-valeur", ascending=False)
display(df_sw.reset_index(drop=True))

### 7.2 Latitude et MAE — y a-t-il une tendance ?

Les villes plus au nord ont-elles des prévisions moins fiables ? C'est une hypothèse intuitive (plus de variabilité météo en zone tempérée nordique).

In [ ]:
df_corr = df_metrics.merge(coords_df[["ville", "lat"]], on="ville")
r, p = stats.pearsonr(df_corr["lat"], df_corr["mae"])

print(f"Corrélation Pearson (latitude × MAE) : r = {r:.3f}, p = {p:.3f}")

if p < 0.05:
    print(f"→ Corrélation significative ({'positive' if r > 0 else 'négative'}).")
else:
    print("→ Pas de corrélation significative sur ce panel (8 villes, n trop faible).")
    print("  Il faudrait un panel plus large pour conclure.")

fig_sc = px.scatter(
    df_corr, x="lat", y="mae", text="ville",
    trendline="ols",
    title=f"Latitude vs MAE — r = {r:.3f} (p = {p:.3f})",
    labels={"lat": "Latitude (°N)", "mae": "MAE (°C)"}
)
fig_sc.update_traces(textposition="top center")
fig_sc.update_layout(template="plotly_white")
fig_sc.show()

---
## 8. Export des résultats

In [ ]:
# CSV
villes_df.to_csv(DATA_PROCESSED_DIR / "villes.csv", index=False)
coords_df.to_csv(DATA_PROCESSED_DIR / "coords.csv", index=False)
df_forecasts.to_csv(DATA_PROCESSED_DIR / "forecasts.csv", index=False)
df_historique.to_csv(DATA_PROCESSED_DIR / "historique.csv", index=False)
df_metrics.to_csv(DATA_PROCESSED_DIR / "metrics.csv", index=False)
df_scores.to_csv(DATA_PROCESSED_DIR / "city_scores.csv", index=False)

# HTML
build_horizon_chart(df_metrics).write_html(OUTPUTS_DIR / "mae_horizon.html")
build_heatmap(df_metrics).write_html(OUTPUTS_DIR / "heatmap_scores.html")
fig_rank.write_html(OUTPUTS_DIR / "ranking_scores.html")
fig_sc.write_html(OUTPUTS_DIR / "lat_vs_mae.html")

print("Export terminé.")
print()
print("CSV dans :", DATA_PROCESSED_DIR)
for f in sorted(DATA_PROCESSED_DIR.glob("*.csv")):
    print(f"  {f.name}")

print()
print("HTML dans :", OUTPUTS_DIR)
for f in sorted(OUTPUTS_DIR.glob("*.html")):
    print(f"  {f.name}")

---
## 9. Bilan

In [ ]:
print("Résultats du pipeline")
print("-" * 50)
print(f"Villes analysées        : {len(villes_df)}")
print(f"Observations / ville    : {df_historique.groupby('ville').size().iloc[0]}")
print(f"Prévisions collectées   : {len(df_forecasts):,}")
print(f"Source données          : {df_forecasts['source'].unique()[0]}")
print()
print(f"MAE moyenne (panel)     : {df_metrics['mae'].mean():.2f} °C")
print(f"RMSE moyen (panel)      : {df_metrics['rmse'].mean():.2f} °C")
print(f"Biais moyen (panel)     : {df_metrics['bias'].mean():+.2f} °C")
print()
top = df_scores.iloc[0]
bot = df_scores.iloc[-1]
print(f"Meilleure fiabilité     : {top['ville']} (score {top['score']:.1f}, MAE {top['mae']:.2f} °C)")
print(f"Fiabilité la plus faible: {bot['ville']} (score {bot['score']:.1f}, MAE {bot['mae']:.2f} °C)")
print()
print("Points à retenir :")
print("  - Les erreurs restent contenues sur ce panel (MAE < 1.6 °C partout)")
print(f"  - Lille présente le biais le plus fort ({df_metrics[df_metrics['ville']=='Lille']['bias'].values[0]:+.2f} °C) : surestimation systématique")
print("  - Pas de corrélation significative latitude/MAE sur 8 villes (panel trop petit)")
print("  - Les données synthétiques valident la méthode, pas les prévisions réelles")

---
## Notes

**Limites principales de ce travail :**

Les données sont synthétiques — l'historique de 30 jours est généré mathématiquement, pas collecté. Les résultats reflètent les propriétés du générateur, pas la météo réelle. Pour passer à quelque chose de plus solide, il faudrait lancer une collecte quotidienne pendant quelques semaines.

Le panel de 8 villes est trop réduit pour des conclusions généralisables. La corrélation latitude/MAE par exemple ne peut pas être testée sérieusement avec si peu de points.

L'analyse porte uniquement sur la température. Les précipitations, le vent et la couverture nuageuse sont des variables au moins aussi importantes pour la plupart des usages pratiques.

**Pistes d'amélioration :**

- Automatiser la collecte quotidienne (GitHub Actions) pour construire un vrai historique
- Étendre le panel à 30–50 villes en incluant des zones de montagne
- Analyser la dégradation de la précision selon l'horizon (J+1 vs J+5)
- Comparer plusieurs fournisseurs de prévisions sur les mêmes villes